In [1]:
import jax
from jax import random
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import time
from dataclasses import dataclass
from jax.scipy.optimize import minimize
import matplotlib.pyplot as plt
from math import exp, pi, sqrt

import jax.numpy as jnp

jax.config.update("jax_enable_x64", True) #for better numerical stability 

number = Union[int, float]
number_like = Union[List[number], number]
array_like = Union[List[number], np.ndarray]
distributions: TypeAlias = Literal["chauchy", "chi2", "expon", "exponpow", "gamma", "lognorm", "norm", "powerlaw", "rayleigh",
                            "uniform", "t", "gumbel_r", "f"]  
FloatArray = npt.NDArray[np.float64]
Floats = Union[float, FloatArray]
Int = Union[int, np.int16, np.int32, np.int64, jnp.int64, jnp.int32, jnp.int16]

from Vares import Auxiliary

#1. потестить на несколько дней вперёд 
#2. разные распределения

In [44]:
@dataclass
class GBMParams:
    volatility: Floats 
    mean: Floats 

def make_gbm_simulator(
        params: GBMParams,
        T: int, 
        granularity: int = 1000
    ): 
    '''Creates a Geometric Brownian Motion simulator with predetermined parameters. 
    Returns a batch of simulated data.'''
    def simulate(n_paths: int, seed = None):
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        time_grid = T * granularity
        dt = 1 / granularity
        norm = rng.normal(size=(n_paths, time_grid))

        #Simulate log returns paths using GBM.
        d_log_S = (
            (params.mean - ((params.volatility**2) / 2)) * dt 
            + params.volatility * norm * np.sqrt(dt)
        )
        d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate 

def _terminal_returns(simulator: callable, n_paths: int, seed = None): 
    '''Helper method for stripping and returning the last column of the return matrix.'''
    cummulative_returns = simulator(n_paths, seed)
    terminal_returns = cummulative_returns[:, -1] #in case of GBM terminal return will be distributed lognormally
    return terminal_returns

In [49]:
class ProtoPortfolio: 
    def __init__(self, returns, **kwargs): 
        self.returns = returns #(portfolio) returns
        try:
            self.number_of_assets = self.returns.shape[1] 
        except IndexError: 
            self.number_of_assets = 1

        self.kwargs = kwargs.copy()

    def calibrate(self, horizon=1, **kwargs): 
        '''Private method implies variance and mean of returns of assets' returns using GARCH(p, q)
        
        Now it is unable to handle returns across multiple(>1) assets. Also can not handle horizon > 1'''
        am = arch_model(self.returns, vol='Garch', dist='normal', **kwargs)
        result = am.fit(disp='off')
        forecast = result.forecast(horizon=horizon)
        mu = result.params.get("mu", 0)
        sigma = forecast.variance.values[0]
        parameters = GBMParams(volatility=sigma, mean=mu)

        self.parameters = parameters #for testing 

        return None
    
    def _simulate(self, T: int = 10, n_paths: int = 500, granularity: int = 1000):
        '''Simualtes GBM n_paths times. Outputs simulated array of terminal returns.'''
        simulator = make_gbm_simulator(self.parameters, T, granularity)
        simulated_returns = _terminal_returns(simulator, n_paths)

        return simulated_returns 
    
    def historical_var(self, alpha=0.01, T: int = 10, **kwargs): 
        _ = self._simulate(T, **kwargs)
        return -np.quantile(_, alpha)




In [ ]:
#forecast for multiple days ahead
def make_gbm_simulator_2(
        params: GBMParams, 
        T: int, 
        granularity: int = 1000 
): 
    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.')
    
    def simulate(n_paths, seed = None): 
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        dt = 1 / granularity

        #version 1
        d_log_S = np.zeros((n_paths, ))
        for day in range(0, T): 
            if day < params.volatility.shape[0]:
                norm = rng.normal(size=(n_paths, granularity))

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * norm * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _
                print(d_log_S.shape)
            else: 
                _ = (
                (params.mean - ((params.volatility[-1]**2) / 2)) * dt 
                + params.volatility[-1] * norm * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1)
        # d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate

            

        
        

In [ ]:
#same as make_gbm_simulator_2 but with array slicing is needed here.

In [123]:
normal_sample = stats.norm.rvs(size=5000)
p = ProtoPortfolio(normal_sample)
p.calibrate(horizon=10)
print(p.parameters.volatility[9])

sim = make_gbm_simulator_2(p.parameters, 11)



0.9657514929619782
Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.


In [125]:
a = sim(200)
print(a.shape)

(200, 1000)
(200, 1000)
(200, 1000)
(200, 2000)
(200, 1000)
(200, 3000)
(200, 1000)
(200, 4000)
(200, 1000)
(200, 5000)
(200, 1000)
(200, 6000)
(200, 1000)
(200, 7000)
(200, 1000)
(200, 8000)
(200, 1000)
(200, 9000)
(200, 1000)
(200, 10000)
(200, 11000)
